In [ ]:
#tight


import cv2
from cvzone.PoseModule import PoseDetector
import math

# --- Helpers ---

def angle_between_points(p1, p2, p3):
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    ang = math.degrees(math.atan2(y3 - y2, x3 - x2) - math.atan2(y1 - y2, x1 - x2))
    ang = abs(ang)
    if ang > 180:
        ang = 360 - ang
    return int(ang)

def safe_lm(lmList, idx):
    if not lmList or idx >= len(lmList):
        return None
    pt = lmList[idx]
    if len(pt) < 2:
        return None
    x, y = int(pt[0]), int(pt[1])
    if x == 0 and y == 0:
        return None
    return (x, y)

def visibility_score(lmList, idx):
    if not lmList or idx >= len(lmList):
        return 0
    pt = lmList[idx]
    if len(pt) >= 3:
        return pt[2]
    return 1

# --- Thresholds and landmark indices ---

LHIP, LKNEE, LANKLE, LFOOT = 23, 25, 27, 31
RHIP, RKNEE, RANKLE, RFOOT = 24, 26, 28, 32
LSHOULDER, RSHOULDER = 11, 12

KNEE_STAND_THRESH = 150      
KNEE_BOTTOM_THRESH = 85      
HIP_STRAIGHT_THRESH = 65    
KNEE_OVER_TOE_PIX = 40 

# --- View-specific processing functions ---

def process_side_view(img, hip_pt, knee_pt, ankle_pt, foot_pt, shoulder_pt, forward_sign,
                      rep_state, rep_broken, counter):
    feedback = []

    hip_angle = angle_between_points(shoulder_pt, hip_pt, knee_pt)
    knee_angle = angle_between_points(hip_pt, knee_pt, ankle_pt)
    ankle_angle = angle_between_points(knee_pt, ankle_pt, foot_pt)

    if forward_sign == 1:
        knee_over_toe = (knee_pt[0] - ankle_pt[0]) > KNEE_OVER_TOE_PIX
    else:
        knee_over_toe = (ankle_pt[0] - knee_pt[0]) > KNEE_OVER_TOE_PIX

    hip_straight = hip_angle > HIP_STRAIGHT_THRESH

    # Feedback on form
    if knee_over_toe:
        feedback.append("Knee past toes")
    if not hip_straight:
        feedback.append("Hip not straight (lean forward)")
    if knee_angle < 30:
        feedback.append("Too deep (risk)")

    if knee_over_toe or (not hip_straight) or (knee_angle < 30):
        rep_broken = True

    # Draw landmarks
    for pt in [hip_pt, knee_pt, ankle_pt, foot_pt, shoulder_pt]:
        cv2.circle(img, pt, 7, (0, 255, 255), cv2.FILLED)

    # Show angles
    cv2.putText(img, f"Hip: {hip_angle}°", (hip_pt[0] + 10, hip_pt[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 215, 0), 2)
    cv2.putText(img, f"Knee: {knee_angle}°", (knee_pt[0] + 10, knee_pt[1] + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 215, 0), 2)
    cv2.putText(img, f"Ankle: {ankle_angle}°", (ankle_pt[0] + 10, ankle_pt[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 215, 0), 2)

    # Rep state machine
    if rep_state == 0:
        if (knee_angle > KNEE_STAND_THRESH) and hip_straight and not knee_over_toe:
            rep_broken = False
            rep_state = 1
    elif rep_state == 1:
        if knee_angle < KNEE_BOTTOM_THRESH:
            rep_state = 2
        elif (knee_angle > KNEE_STAND_THRESH) or rep_broken:
            rep_state = 0
    elif rep_state == 2:
        if knee_angle > KNEE_BOTTOM_THRESH:
            rep_state = 3
        elif rep_broken:
            rep_state = 0
    elif rep_state == 3:
        if (knee_angle > KNEE_STAND_THRESH) and hip_straight and not knee_over_toe:
            if not rep_broken:
                counter += 1
            rep_state = 0
            rep_broken = False
        elif rep_broken:
            rep_state = 0
            rep_broken = False

    return rep_state, rep_broken, counter, feedback, "Side View"


def process_front_view(img, hip_pt, knee_pt, ankle_pt, shoulder_pt,
                       rep_state, rep_broken, counter):
    feedback = []

    hip_angle = angle_between_points(shoulder_pt, hip_pt, knee_pt)
    knee_angle = angle_between_points(hip_pt, knee_pt, ankle_pt)

    hip_straight = hip_angle > HIP_STRAIGHT_THRESH

    if not hip_straight:
        feedback.append("Hip not straight")

    is_standing = knee_angle > KNEE_STAND_THRESH and hip_straight
    is_bottom = knee_angle < KNEE_BOTTOM_THRESH

    if not is_standing and not is_bottom:
        rep_broken = True

    # Draw landmarks
    for pt in [hip_pt, knee_pt, ankle_pt, shoulder_pt]:
        cv2.circle(img, pt, 7, (255, 255, 0), cv2.FILLED)

    # Show angles
    cv2.putText(img, f"Hip: {hip_angle}°", (hip_pt[0] + 10, hip_pt[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 215, 0), 2)
    cv2.putText(img, f"Knee: {knee_angle}°", (knee_pt[0] + 10, knee_pt[1] + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 215, 0), 2)

    # State machine
    if rep_state == 0:
        if is_standing:
            rep_broken = False
            rep_state = 1
    elif rep_state == 1:
        if is_bottom:
            rep_state = 2
        elif not is_standing or rep_broken:
            rep_state = 0
    elif rep_state == 2:
        if not is_bottom:
            rep_state = 3
        elif rep_broken:
            rep_state = 0
    elif rep_state == 3:
        if is_standing:
            if not rep_broken:
                counter += 1
            rep_state = 0
            rep_broken = False
        elif rep_broken:
            rep_state = 0
            rep_broken = False

    return rep_state, rep_broken, counter, feedback, "Front View"


# --- Main program ---

# Change this variable to:
# "front" for front view only
# "side" for side view only
# "auto" for automatic detection (both)
VIEW_MODE = "side"

cap = cv2.VideoCapture("D:/University/Fyp Ideas/How To Do Perfect SQUAT _ FITNESS SPECIAL _ SQUATS For Beginners _ WORKOUT VIDEO.mp4")
# cap = cv2.VideoCapture(0)
detector = PoseDetector(detectionCon=0.6, trackCon=0.6)

counter = 0
rep_state = 0
rep_broken = False

cv2.namedWindow("Squat Detector", cv2.WND_PROP_FULLSCREEN)
cv2.setWindowProperty("Squat Detector", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

while True:
    ret, img = cap.read()
    if not ret:
        break

    # No flip: raw feed

    detector.findPose(img, draw=True)
    lmList, _ = detector.findPosition(img, draw=False)

    feedback = []
    view_mode = "No pose detected"

    if lmList:
        r_vis = [visibility_score(lmList, idx) for idx in [RHIP, RKNEE, RANKLE, RFOOT, RSHOULDER]]
        l_vis = [visibility_score(lmList, idx) for idx in [LHIP, LKNEE, LANKLE, LFOOT, LSHOULDER]]

        if sum(r_vis) > sum(l_vis):
            side = 'right'
            hip_pt = safe_lm(lmList, RHIP)
            knee_pt = safe_lm(lmList, RKNEE)
            ankle_pt = safe_lm(lmList, RANKLE)
            foot_pt = safe_lm(lmList, RFOOT)
            shoulder_pt = safe_lm(lmList, RSHOULDER)
            forward_sign = 1
        else:
            side = 'left'
            hip_pt = safe_lm(lmList, LHIP)
            knee_pt = safe_lm(lmList, LKNEE)
            ankle_pt = safe_lm(lmList, LANKLE)
            foot_pt = safe_lm(lmList, LFOOT)
            shoulder_pt = safe_lm(lmList, LSHOULDER)
            forward_sign = -1

        # Now handle based on VIEW_MODE
        if VIEW_MODE == "front":
            if hip_pt and knee_pt and ankle_pt and shoulder_pt:
                rep_state, rep_broken, counter, feedback, view_mode = process_front_view(
                    img, hip_pt, knee_pt, ankle_pt, shoulder_pt,
                    rep_state, rep_broken, counter)
            else:
                view_mode = "Insufficient landmarks"

        elif VIEW_MODE == "side":
            if hip_pt and knee_pt and ankle_pt and foot_pt and shoulder_pt:
                rep_state, rep_broken, counter, feedback, view_mode = process_side_view(
                    img, hip_pt, knee_pt, ankle_pt, foot_pt, shoulder_pt, forward_sign,
                    rep_state, rep_broken, counter)
            else:
                view_mode = "Insufficient landmarks"

        else:  # auto mode
            if hip_pt and knee_pt and ankle_pt and foot_pt and shoulder_pt:
                horiz_dist = abs(knee_pt[0] - ankle_pt[0])
                if horiz_dist > 15:
                    rep_state, rep_broken, counter, feedback, view_mode = process_side_view(
                        img, hip_pt, knee_pt, ankle_pt, foot_pt, shoulder_pt, forward_sign,
                        rep_state, rep_broken, counter)
                else:
                    rep_state, rep_broken, counter, feedback, view_mode = process_front_view(
                        img, hip_pt, knee_pt, ankle_pt, shoulder_pt,
                        rep_state, rep_broken, counter)
            else:
                view_mode = "Insufficient landmarks"

    # Display info and feedback
    cv2.putText(img, f"Currently: {view_mode}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(img, f"Reps: {counter}", (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    y = 110
    for fb in feedback:
        cv2.putText(img, fb, (20, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        y += 30

    cv2.imshow("Squat Detector", img)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q') or key == 27:
        break

cap.release()
cv2.destroyAllWindows()
